# Import Library

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
import seaborn as sns
import matplotlib.pyplot as plt
from joblib import dump

# Memanggil dataset

In [ ]:
df = pd.read_csv("datasetmobil.csv")

# Deskripsi dataset

In [ ]:
#Pisahkan fitur dan target
X = df.drop(columns="harga")
y = df["harga"]

# Tampilkan hasil pemisahan
print("=== Fitur (X) ===")
print(X.head())

print("\n=== Target (y) ===")
print(y.head())

In [ ]:
df.info()

In [ ]:
df.describe

# Definisikan kolom kategorikal dan numerikal

In [ ]:
categorical_cols = ["model", "transmisi", "bahan_bakar"]
numerical_cols = ["tahun", "kilometer", "pajak", "mpg", "cc"]

# Tampilkan hasil identifikasi
print("=== Kolom Kategorikal ===")
print(categorical_cols)

print("\n=== Kolom Numerikal ===")
print(numerical_cols)

# Correlation Matrix

In [ ]:
# Select only the numerical columns and the target column
df_numerical_target = df[numerical_cols + ["harga"]]

# Calculate the correlation matrix
correlation_matrix = df_numerical_target.corr()

# Plot the correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix of Numerical Features and Target")
plt.show()

# Preprocessing

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ],
    remainder="passthrough"
)

# Fit dan transformasikan data (contoh menggunakan sebagian data untuk tampilan)
X_transformed = preprocessor.fit_transform(X)

# Untuk melihat hasil transformasi, kita ubah ke DataFrame
# Ambil nama fitur hasil one-hot encoding
encoded_feature_names = preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_cols)

# Gabungkan dengan nama kolom numerik
all_feature_names = list(encoded_feature_names) + numerical_cols

# Buat DataFrame dari hasil transformasi
import pandas as pd
X_transformed_df = pd.DataFrame(X_transformed.toarray() if hasattr(X_transformed, "toarray") else X_transformed,
                                columns=all_feature_names)

# Tampilkan hasil preprocessing
print("=== Data Setelah Preprocessing ===")
print(X_transformed_df.head())

# Pipeline regresi dengan RandomForest

In [ ]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(random_state=42))
])

# Tampilkan isi pipeline
print("=== Struktur Pipeline ===")
for name, step in pipeline.steps:
    print(f"- {name}: {type(step).__name__}")

# Split data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Tampilkan ukuran data train dan test
print("=== Ukuran Data Setelah Split ===")
print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test : {y_test.shape}")

# Latih model

In [ ]:
pipeline.fit(X_train, y_train)

# Evaluasi model

In [ ]:
y_pred = pipeline.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred) * 100  # dikalikan 100 agar dalam persentase

# Tampilkan hasil evaluasi
print("=== Evaluasi Model ===")
print(f"Mean Squared Error (MSE)               : {mse:,.2f}")
print(f"R-squared (R?)                         : {r2:.4f}")
print(f"Mean Absolute Percentage Error (MAPE) : {mape:.2f}%")

In [ ]:
input_data = pd.DataFrame([{
    'model': 'Cayla',
    'tahun': 2021,
    'transmisi': 'manual',
    'kilometer': 165188200,
    'bahan_bakar': 'bensin',
    'pajak': 2500012,
    'mpg': 40.3,
    'cc': 1.5
}])

prediction = pipeline.predict(input_data)[0]
print(f"\nHarga mobil (dalam Rupiah): Rp{prediction:,.0f}")

In [ ]:
prediction = pipeline.predict(input_data)[0]
print(f"\nEstimasi harga mobil (dalam Rupiah): Rp{prediction:,.0f}")

# Save model

In [ ]:
dump(pipeline, 'harga_mobil.joblib')
print("\nModel berhasil disimpan sebagai 'harga_mobil.joblib'")
